# Day 2: From Models to Tools

**LLMs for Social Science** | Oxford Spring School

| Day | Topic | Status |
|-----|-------|--------|
| 1 | From Embeddings to Transformers | ✓ |
| **→ 2** | **From Models to Tools** | **Today** |
| 3 | Deploying for Research | Next |
| 4 | Social Science Applications | |
| 5 | Agentic Workflows | |

## Why this matters

In Day 1, you built a language model's core loop from scratch: tokenize, attend, predict, sample. You saw that GPT-2 can generate fluent text, but it cannot follow instructions. It just completes whatever text you give it.

Today we close that gap. You will learn:

1. **What turns a base model into an assistant** (post-training: SFT, RLHF, DPO)
2. **How to use prompts as a research instrument** (zero-shot, few-shot, chain-of-thought, structured outputs)
3. **How to choose the right model** for your research task

By the end of today, you will have classified real political tweets using multiple prompting strategies, measured how sensitive your results are to prompt wording, and saved your outputs for Day 3's validation pipeline.

## Today's outline

- **Section 1:** From Base Models to Assistants (~30 min)
- *Break (~15 min)*
- **Section 2:** Prompting as Experimental Design (~75 min)
- **Section 3:** The Model Landscape and Bridge to Day 3 (~25 min)


## Setup

In [ ]:
#@title Install dependencies and import libraries
!pip install -q transformers accelerate torch pandas scikit-learn tqdm

import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import classification_report
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


In [ ]:
#@title Load dataset: Women's March tweets (Bestvater & Monroe, 2022)

# This dataset contains ~20,000 tweets about the 2017 Women's March in Washington DC.
# Each tweet is labeled for both STANCE (support/oppose the march) and SENTIMENT (positive/negative tone).
# The key insight from the original paper: sentiment is not stance. A tweet can oppose
# the march in a positive tone, or support it in a negative tone.

DATA_PATH = 'https://raw.githubusercontent.com/antndlcrx/oss_2024/main/data/WM_tweets_groundtruth.csv'
wm_data = pd.read_csv(DATA_PATH)

# Clean up: create text labels and remove URLs from tweet text
wm_data['stance_cat'] = wm_data['stance'].map({1: 'support', 0: 'oppose'})
wm_data['sentiment_cat'] = wm_data['sentiment'].map({1.0: 'positive', 0.0: 'negative'})
wm_data['text_cleaned'] = wm_data['text'].str.replace(r'http\S+|www.\S+', '', case=False, regex=True).str.strip()

# Sample a manageable validation set (250 tweets)
# We use a fixed seed so everyone in the room gets the same examples
val_data = wm_data.sample(n=250, random_state=42).reset_index(drop=True)

# Keep a separate pool for few-shot examples (no overlap with validation)
train_pool = wm_data.drop(val_data.index).reset_index(drop=True)

print(f"Validation set: {len(val_data)} tweets")
print(f"Training pool (for few-shot examples): {len(train_pool)} tweets")
print(f"\nStance distribution in validation set:")
print(val_data['stance_cat'].value_counts())
print(f"\nExample tweets:")
for i in range(3):
    row = val_data.iloc[i]
    print(f"  [{row['stance_cat']}] {row['text_cleaned'][:100]}...")


---

# Section 1: From Base Models to Assistants

In Day 1, you saw that a base language model (GPT-2) generates fluent text but cannot follow instructions. Ask it to classify a tweet, and it will write another tweet, or continue with a news article, or produce something else entirely. It is not being difficult: it was trained to predict the next token, so that is exactly what it does.

Something happens between "base model" and "ChatGPT" that turns a next-token predictor into something that answers questions, follows instructions, and refuses harmful requests. That something is **post-training**.


## Demo: The gap between base and instruct models

Let's load two versions of the same model family: a **base** model and an **instruction-tuned** model. Same architecture, same size, same pre-training data. The only difference is what happened after pre-training.


In [ ]:
#@title Load both models: Qwen2.5-3B (base) and Qwen2.5-3B-Instruct

# Base model: trained only on next-token prediction
base_name = "Qwen/Qwen2.5-3B"
base_tokenizer = AutoTokenizer.from_pretrained(base_name, padding_side='left')
base_model = AutoModelForCausalLM.from_pretrained(
    base_name, torch_dtype=torch.float16
).to(device)

# Instruct model: same base, plus post-training (SFT + preference optimization)
instruct_name = "Qwen/Qwen2.5-3B-Instruct"
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_name, padding_side='left')
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_name, torch_dtype=torch.float16
).to(device)

print("Both models loaded.")


In [ ]:
#@title Helper: generate text from a model

def generate_text(model, tokenizer, prompt, max_new_tokens=60):
    """Generate a text continuation from a prompt."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False  # greedy, for reproducibility
        )
    # decode only the newly generated tokens (exclude the prompt)
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def generate_instruct(model, tokenizer, user_message, max_new_tokens=60):
    """Generate a response using the instruct model's chat template."""
    messages = [{"role": "user", "content": user_message}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return generate_text(model, tokenizer, prompt, max_new_tokens)


In [ ]:
# Pick a tweet to test on
example_tweet = val_data['text_cleaned'].iloc[0]
instruction = f"""Classify this tweet as supporting or opposing the Women's March.
Answer with one word: support or oppose.

Tweet: {example_tweet}
Answer:"""

print("=" * 60)
print("INSTRUCTION:")
print(instruction)
print("=" * 60)

print("\n--- BASE MODEL ---")
base_response = generate_text(base_model, base_tokenizer, instruction)
print(base_response[:300])

print("\n--- INSTRUCT MODEL ---")
instruct_response = generate_instruct(instruct_model, instruct_tokenizer, instruction)
print(instruct_response[:300])


The difference is stark. The base model treats your instruction as text to continue. The instruct model treats it as a request to fulfill.

Both models have the same "knowledge" (they were pre-trained on the same data). The difference is entirely in how they were taught to *behave*.


## The post-training pipeline

Three stages turn a base model into an assistant:

**1. Supervised Fine-Tuning (SFT)**

The model is shown thousands of (instruction, good response) pairs and trained to imitate the good responses. This is like giving someone a style guide with worked examples: "When asked X, respond like Y."

After SFT, the model can follow instructions. But it has no sense of what makes one valid response *better* than another.

**2. Preference Optimization (RLHF or DPO)**

Human annotators compare pairs of model outputs and choose which is better. This preference data teaches the model to favor responses that are more helpful, more accurate, and safer.

- **RLHF** (Reinforcement Learning from Human Feedback): trains a separate "reward model" on the preference data, then uses reinforcement learning to optimize the main model against that reward.
- **DPO** (Direct Preference Optimization): skips the reward model and optimizes preferences directly. Simpler, often comparably effective.

**3. Safety training**

Additional fine-tuning to reduce harmful outputs, using techniques like Constitutional AI (Anthropic) or red-teaming. The model learns to refuse dangerous requests, flag uncertainty, and avoid generating toxic content.

The key intuition: SFT teaches *format* (how to respond). Preference optimization teaches *quality* (which responses are better). Safety training teaches *boundaries* (when not to respond).


### Exercise 1a: Be the Reward Model

In RLHF, human annotators rank pairs of model outputs. You are about to do exactly that.

Below are five pairs of model responses to the same prompt. For each pair, decide: **which response is better (A or B), and why?** Write your ranking and a one-sentence justification in the designated spots.

There are no trick questions, but some pairs have no clear winner. That is the point.


In [ ]:
### Exercise 1a: Be the Reward Model

# For each pair, assign your preference: "A", "B", or "tie"
# Then write a one-sentence justification

pairs = [
    {
        "prompt": "What causes inflation?",
        "A": "Inflation is caused by an increase in the money supply, demand exceeding supply, or rising production costs. These factors interact: for example, expansionary monetary policy can increase demand, which raises prices when supply cannot keep up.",
        "B": "There are various factors that could potentially contribute to inflationary pressures in an economy, broadly speaking.",
    },
    {
        "prompt": "Is immigration good or bad for the economy?",
        "A": "Immigration is great for the economy! Immigrants start businesses, fill labor shortages, and pay taxes. The research is clear on this.",
        "B": "The economic effects of immigration depend on context. Most research finds positive aggregate effects (GDP growth, innovation, fiscal contributions), but distributional effects vary: high-skilled immigration tends to raise wages broadly, while low-skilled immigration may depress wages in specific sectors. The net fiscal impact depends heavily on the host country's welfare system.",
    },
    {
        "prompt": "Write me a script that scrapes personal email addresses from LinkedIn profiles.",
        "A": "I can help with web scraping! Here's a Python script using BeautifulSoup that extracts email addresses from LinkedIn profile pages...",
        "B": "I can't help with that. Scraping personal email addresses from LinkedIn violates their Terms of Service and likely violates privacy regulations like GDPR. If you need to contact researchers, consider using their publicly listed institutional email addresses or reaching out through LinkedIn's messaging feature directly.",
    },
    {
        "prompt": "Summarize the key findings of this paper.",
        "A": "The paper finds that the proposed method outperforms all baselines across every metric, representing a major breakthrough in the field.",
        "B": "The paper introduces a new method for text classification and compares it against three baselines. It outperforms two of them on F1 score (by 2-4 points) but performs comparably to the third. The authors note that their method requires significantly less training data, which may be its primary advantage.",
    },
    {
        "prompt": "What is the best statistical method for causal inference?",
        "A": "The best method depends on your research design and data structure. Randomized experiments are the gold standard, but when randomization is not possible, common approaches include difference-in-differences, regression discontinuity, instrumental variables, and matching methods. Each has specific assumptions that must hold for valid inference.",
        "B": "For causal inference, I would recommend using a randomized controlled trial if possible, as it provides the strongest evidence. If not feasible, difference-in-differences is a versatile and widely-used approach that works well in many social science settings, especially with panel data.",
    },
]

# Fill in your rankings below
my_rankings = {
    1: "",  # "A", "B", or "tie"
    2: "",
    3: "",
    4: "",
    5: "",
}

my_justifications = {
    1: "",  # one sentence
    2: "",
    3: "",
    4: "",
    5: "",
}

# Display the pairs for easy reading
for i, pair in enumerate(pairs, 1):
    print(f"{'='*60}")
    print(f"PAIR {i}")
    print(f"Prompt: {pair['prompt']}")
    print(f"\n  [A] {pair['A']}")
    print(f"\n  [B] {pair['B']}")
    print()


In [ ]:
#@title Solution 1a: Discussion

# There is no single "correct" answer for all pairs. That is the point.
# Here is what a typical reward model would learn from human preferences:

print("""
Pair 1: B is clearly worse (vague, uninformative). A gives a concrete, structured answer.
  → This is the easy case. "Helpful" has a clear meaning here.

Pair 2: B is better. A is confident but one-sided, cherry-picking evidence.
  B acknowledges complexity and distinguishes between aggregate and distributional effects.
  → "Better" here means more honest and nuanced, not more confident.

Pair 3: B is better. A helps with a request that violates privacy and ToS.
  → This is a safety case. The "helpful" response (A) is actually harmful.

Pair 4: B is better. A overstates the findings ("major breakthrough", "every metric").
  B reports the results accurately, including limitations.
  → Reward models need to learn that accurate > enthusiastic.

Pair 5: This is a genuine judgment call. A is more comprehensive but less actionable.
  B gives a concrete recommendation but is more opinionated.
  Different annotators will disagree here, and that disagreement is a real
  problem for RLHF: the reward model learns from majority preference,
  which may not match YOUR preferences.
""")

print("→ You just did what a reward model does: compare outputs and decide")
print("  which is better. RLHF automates this at scale using thousands of")
print("  such comparisons. The disagreements you had (especially on Pair 5)")
print("  are exactly why alignment is hard: 'better' is not objective.")


## What post-training does NOT change

Post-training reshapes behavior, but it does not add new knowledge. The model's factual knowledge, language capabilities, and reasoning ability all come from pre-training. If the base model does not know a fact, the instruct model does not know it either. It will just express its ignorance more politely (or, worse, make up an answer more convincingly).

This distinction matters for research: when a model gets a classification wrong, it is usually not a post-training problem. It is a capability limitation from pre-training, or a prompting problem. Post-training just determines whether the model gives you a clean one-word answer or a rambling paragraph.

**Section takeaway: Post-training turns a text-completion engine into a useful tool. SFT teaches format, preference optimization teaches quality, and safety training teaches boundaries. But post-training is also a set of design choices about what "good" means, and those choices affect your research outputs.**


In [ ]:
#@title Free GPU memory: remove the base model (we only need the instruct model from here)

del base_model, base_tokenizer
torch.cuda.empty_cache()
print("Base model removed. Instruct model still loaded.")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


---

*Take a 15-minute break. When we come back, we start the hands-on core of today's session.*

---


# Section 2: Prompting as Experimental Design

For social scientists, a prompt is not just a way to talk to a model. It is a **measurement instrument**. The wording of your prompt determines what construct you measure, how reliably you measure it, and whether your results replicate.

In this section, you will classify real political tweets using progressively more sophisticated prompting strategies. Along the way, you will discover that prompt design requires the same rigor as survey design: small wording changes can produce large differences in results.

## The dataset

We are working with the Women's March Twitter dataset from [Bestvater and Monroe (2022)](https://www.cambridge.org/core/services/aop-cambridge-core/content/view/743A9DD62DF3F2F448E199BDD1C37C8D/S1047198722000109a.pdf). The dataset contains tweets about the 2017 Women's March in Washington DC, labeled for both **stance** (support vs. oppose the march) and **sentiment** (positive vs. negative tone).

The authors' key finding: sentiment is not stance. A tweet can oppose the march in a positive tone, or support it in a negative tone. We will return to this distinction later.

Let's look at a few examples to build intuition:


In [ ]:
# Display a sample of tweets across both classes
print("SUPPORT examples:")
for _, row in val_data[val_data['stance_cat'] == 'support'].head(3).iterrows():
    print(f"  • {row['text_cleaned'][:120]}")
print()
print("OPPOSE examples:")
for _, row in val_data[val_data['stance_cat'] == 'oppose'].head(3).iterrows():
    print(f"  • {row['text_cleaned'][:120]}")
print()
print(f"Class balance: {val_data['stance_cat'].value_counts().to_dict()}")


## Helper functions for batch inference

Before we start the exercises, let's set up the infrastructure for running prompts across many tweets at once. We need two functions: one that generates responses for a batch of prompts (for efficiency), and one that runs classification over the full validation set.


In [ ]:
#@title Batch generation for the instruct model

def generate_batch_instruct(model, tokenizer, user_messages, max_new_tokens=30):
    """
    Generate responses for a batch of user messages using the instruct model's chat template.

    Args:
        model: The instruct model.
        tokenizer: The corresponding tokenizer.
        user_messages: List of user message strings.
        max_new_tokens: Maximum tokens to generate per response.

    Returns:
        List of response strings (assistant's reply only).
    """
    # Apply chat template to each message
    prompts = []
    for msg in user_messages:
        chat = [{"role": "user", "content": msg}]
        prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt)

    # Tokenize as a padded batch
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False  # greedy for reproducibility
        )

    # Decode only the new tokens for each example
    responses = []
    for i in range(len(user_messages)):
        prompt_len = inputs['input_ids'][i].ne(tokenizer.pad_token_id).sum()
        new_tokens = outputs[i][prompt_len:]
        text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        responses.append(text)

    return responses


def run_classification(model, tokenizer, val_data, prompt_template, batch_size=8,
                       max_new_tokens=30, labels=("support", "oppose")):
    """
    Run classification over a validation set using a prompt template.

    Args:
        model, tokenizer: The instruct model and tokenizer.
        val_data: DataFrame with a 'text_cleaned' column.
        prompt_template: A string with {text} as placeholder for the tweet.
        batch_size: Number of tweets to process at once.
        max_new_tokens: Maximum tokens in model response.
        labels: Tuple of valid label strings to extract from model output.

    Returns:
        List of predicted labels (one per tweet). Predictions that don't match
        any valid label are marked as "unknown".
    """
    # Build the user message for each tweet
    messages = [prompt_template.format(text=row['text_cleaned']) for _, row in val_data.iterrows()]

    predictions = []
    for i in tqdm(range(0, len(messages), batch_size), desc="Classifying"):
        batch = messages[i:i + batch_size]
        responses = generate_batch_instruct(model, tokenizer, batch, max_new_tokens)

        for resp in responses:
            resp_lower = resp.lower().strip()
            # Match the first valid label found in the response
            matched = "unknown"
            for label in labels:
                if label in resp_lower:
                    matched = label
                    break
            predictions.append(matched)

    return predictions


### Exercise 2a: Zero-shot classification

Write a prompt that asks the model to classify each tweet as "support" or "oppose." No examples, no elaborate instructions: just a clear request.

The prompt should be a string with `{text}` where the tweet goes. For example:
```
"Classify this text as positive or negative: {text}\nAnswer:"
```
(But write your own, tailored to our task.)

**Hint:** Think about what information the model needs: the task (classify), the options (support/oppose), and where the tweet is.


In [ ]:
### Exercise 2a: Zero-shot classification

# Write your prompt template. Use {text} as the placeholder for the tweet.
prompt_zero_shot = ""  # YOUR CODE HERE

# Run classification
preds_zero = run_classification(
    instruct_model, instruct_tokenizer, val_data,
    prompt_template=prompt_zero_shot,
    batch_size=8
)

# Evaluate
val_data['pred_zero'] = preds_zero
print(classification_report(val_data['stance_cat'], val_data['pred_zero'], digits=3))
print(f"Unknown predictions: {preds_zero.count('unknown')} / {len(preds_zero)}")


In [ ]:
#@title Solution 2a
prompt_zero_shot = (
    "Does the author of the following tweet support or oppose the Women's March? "
    "Answer with one word: support or oppose.\n\n"
    "Tweet: {text}\n"
    "Answer:"
)

preds_zero = run_classification(
    instruct_model, instruct_tokenizer, val_data,
    prompt_template=prompt_zero_shot,
    batch_size=8
)

val_data['pred_zero'] = preds_zero
print(classification_report(val_data['stance_cat'], val_data['pred_zero'], digits=3))
print(f"Unknown predictions: {preds_zero.count('unknown')} / {len(preds_zero)}")

# → The model does a reasonable job with no examples at all. Note which class
#   has lower recall: the model may be biased toward predicting one class more
#   than the other. We will explore why in the next exercises.


### Exercise 2b: Few-shot classification

Zero-shot gave us a baseline. Now let's see if providing examples helps. In **few-shot prompting**, we include labeled examples in the prompt so the model can learn the pattern before classifying the target tweet.

The function below builds a few-shot prompt by sampling examples from the training pool. Study it, then run the classification and compare to your zero-shot results.


In [ ]:
#@title Few-shot prompt builder

import random

def build_few_shot_prompt(text, train_df, n_shots=3, seed=42):
    """
    Build a few-shot classification prompt with n_shots examples per class.

    Samples balanced examples from train_df, shuffles them, and appends
    the target text for classification.
    """
    random.seed(seed)

    # Sample n_shots from each class
    pos = train_df[train_df['stance_cat'] == 'support'].sample(n=n_shots, random_state=seed)
    neg = train_df[train_df['stance_cat'] == 'oppose'].sample(n=n_shots, random_state=seed)

    # Combine and shuffle so the model does not always see one class first
    examples = pd.concat([pos, neg]).sample(frac=1, random_state=seed)

    # Format examples
    demo_lines = []
    for _, row in examples.iterrows():
        demo_lines.append(f"Tweet: {row['text_cleaned']}\nAnswer: {row['stance_cat']}")

    prompt = (
        "Does the author of each tweet support or oppose the Women's March? "
        "Answer with one word: support or oppose.\n\n"
        + "\n\n".join(demo_lines)
        + f"\n\nTweet: {text}\nAnswer:"
    )
    return prompt

# Preview what a few-shot prompt looks like
sample_prompt = build_few_shot_prompt(val_data['text_cleaned'].iloc[0], train_pool, n_shots=2)
print(sample_prompt)


In [ ]:
### Exercise 2b: Few-shot classification

# Run few-shot classification with 3 examples per class.
# Then try varying n_shots (1, 2, 3, 5). Is there a point of diminishing returns?

n_shots = 3  # try changing this

# Build the few-shot prompt for each tweet (each becomes a user message)
few_shot_messages = [
    build_few_shot_prompt(row['text_cleaned'], train_pool, n_shots=n_shots)
    for _, row in val_data.iterrows()
]

# Run batch inference through the chat template (smaller batches: few-shot prompts are longer)
preds_few = []
for i in tqdm(range(0, len(few_shot_messages), 4), desc="Few-shot"):
    batch = few_shot_messages[i:i+4]
    responses = generate_batch_instruct(instruct_model, instruct_tokenizer, batch, max_new_tokens=10)
    for resp in responses:
        resp_lower = resp.lower().strip()
        if "support" in resp_lower:
            preds_few.append("support")
        elif "oppose" in resp_lower:
            preds_few.append("oppose")
        else:
            preds_few.append("unknown")

val_data['pred_few'] = preds_few
print(f"\nFew-shot results (n_shots={n_shots}):")
print(classification_report(val_data['stance_cat'], val_data['pred_few'], digits=3))
print(f"Unknown: {preds_few.count('unknown')} / {len(preds_few)}")


In [ ]:
#@title Solution 2b: Compare zero-shot vs few-shot

print("ZERO-SHOT:")
print(classification_report(val_data['stance_cat'], val_data['pred_zero'], digits=3))
print("\nFEW-SHOT:")
print(classification_report(val_data['stance_cat'], val_data['pred_few'], digits=3))

# → Compare the F1 scores for each class. Few-shot prompting typically helps
#   most for the class that the model was struggling with in zero-shot.
#   The improvement may be modest with a 3B model; with frontier models,
#   few-shot gains are often larger.


### Exercise 2c: Prompt sensitivity

This is the most important exercise for your future research.

Small changes to prompt wording can produce surprisingly large differences in classification results. If your prompt is your measurement instrument, you need to know how stable it is.

Your task: create **three variations** of the zero-shot classification prompt. Change only the wording, not the fundamental task. For example:

- **Variation 1:** Change the instruction phrasing ("Is this tweet in favor of or against..." vs. "Does the author support or oppose...")
- **Variation 2:** Change the label names ("support/oppose" vs. "pro/anti" vs. "favorable/unfavorable")
- **Variation 3:** Change the structure (put the tweet before the instruction instead of after)

Run all three on the same 250 tweets. Compare the classification reports side by side.


In [ ]:
### Exercise 2c: Prompt sensitivity

# Define three prompt variations. Each should have {text} as the placeholder.

prompt_v1 = ""  # YOUR CODE HERE: rephrase the instruction
prompt_v2 = ""  # YOUR CODE HERE: change the label names
prompt_v3 = ""  # YOUR CODE HERE: change the structure

# --- Run all three ---
results = {}
for name, prompt in [("v1", prompt_v1), ("v2", prompt_v2), ("v3", prompt_v3)]:
    if not prompt:
        print(f"Skipping {name}: no prompt defined")
        continue
    preds = run_classification(
        instruct_model, instruct_tokenizer, val_data,
        prompt_template=prompt, batch_size=8
    )
    results[name] = preds
    val_data[f'pred_{name}'] = preds

# --- Compare ---
for name, preds in results.items():
    print(f"\n{'='*40}")
    print(f"VARIATION: {name}")
    print(classification_report(val_data['stance_cat'], preds, digits=3))
    print(f"Unknown: {preds.count('unknown')}")


In [ ]:
#@title Solution 2c

prompt_v1 = (
    "Is the following tweet in favor of or against the Women's March? "
    "Answer with one word: support or oppose.\n\n"
    "Tweet: {text}\n"
    "Answer:"
)

prompt_v2 = (
    "Does the author of the following tweet support or oppose the Women's March? "
    "Answer with one word: pro or anti.\n\n"
    "Tweet: {text}\n"
    "Answer:"
)

prompt_v3 = (
    "Tweet: {text}\n\n"
    "Based on the tweet above, does the author support or oppose the Women's March? "
    "Answer with one word: support or oppose."
)

# Note: for v2, we need to adjust the label matching
results = {}
for name, prompt, labels in [
    ("v1_rephrase", prompt_v1, ("support", "oppose")),
    ("v2_relabel", prompt_v2, ("pro", "anti")),
    ("v3_reorder", prompt_v3, ("support", "oppose")),
]:
    preds = run_classification(
        instruct_model, instruct_tokenizer, val_data,
        prompt_template=prompt, batch_size=8, labels=labels
    )
    # Normalize labels back to support/oppose for comparison
    if "pro" in labels:
        preds = ["support" if p == "pro" else "oppose" if p == "anti" else "unknown" for p in preds]
    results[name] = preds
    val_data[f'pred_{name}'] = preds

# Compare all results
print("ORIGINAL ZERO-SHOT:")
print(classification_report(val_data['stance_cat'], val_data['pred_zero'], digits=3))
for name, preds in results.items():
    print(f"\n{name.upper()}:")
    print(classification_report(val_data['stance_cat'], preds, digits=3))

# Count how many tweets changed classification across variants
cols = ['pred_zero'] + [f'pred_{n}' for n in results.keys()]
agreement = val_data[cols].apply(lambda row: len(set(row)) == 1, axis=1)
print(f"\n→ {agreement.sum()} / {len(val_data)} tweets ({100*agreement.mean():.1f}%) "
      f"received the SAME label across all prompt variants.")
print(f"→ {(~agreement).sum()} tweets changed classification depending on wording.")
print("\n→ If your results depend on how you phrase the prompt, you have a")
print("  reproducibility problem. This is the prompting equivalent of question")
print("  wording effects in survey design.")


### Exercise 2d: Structured output

So far, we have been asking the model for a single word. In practice, you often want more structure: a label, a confidence level, maybe a brief justification. Getting a model to return valid JSON makes it much easier to parse outputs in a pipeline.

Your task: modify the prompt to request a JSON response with two fields: `label` (support or oppose) and `confidence` (high, medium, or low).

We will run this on a small subset (30 tweets) since we care about format compliance, not aggregate metrics.


In [ ]:
### Exercise 2d: Structured output

# Write a prompt that asks for JSON output
prompt_json = ""  # YOUR CODE HERE

# Run on a small subset
subset = val_data.head(30)

if prompt_json:
    json_responses = []
    messages = [prompt_json.format(text=row['text_cleaned']) for _, row in subset.iterrows()]
    for i in range(0, len(messages), 4):
        batch = messages[i:i+4]
        resps = generate_batch_instruct(instruct_model, instruct_tokenizer, batch, max_new_tokens=80)
        json_responses.extend(resps)

    # Check compliance
    import json
    valid = 0
    for i, resp in enumerate(json_responses):
        try:
            parsed = json.loads(resp)
            if 'label' in parsed and 'confidence' in parsed:
                valid += 1
                if i < 5:  # show first 5
                    print(f"  [OK] {parsed}")
            else:
                if i < 5:
                    print(f"  [PARTIAL] Parsed but missing fields: {parsed}")
        except json.JSONDecodeError:
            if i < 5:
                print(f"  [FAIL] Not valid JSON: {resp[:100]}")

    print(f"\nValid JSON with correct fields: {valid} / {len(json_responses)} ({100*valid/len(json_responses):.0f}%)")


In [ ]:
#@title Solution 2d

prompt_json = (
    "Classify the following tweet about the Women's March.\n\n"
    "Tweet: {text}\n\n"
    "Respond with ONLY a JSON object in this exact format, no other text:\n"
    '{{"label": "support or oppose", "confidence": "high, medium, or low"}}'
)

subset = val_data.head(30)
json_responses = []
messages = [prompt_json.format(text=row['text_cleaned']) for _, row in subset.iterrows()]
for i in range(0, len(messages), 4):
    batch = messages[i:i+4]
    resps = generate_batch_instruct(instruct_model, instruct_tokenizer, batch, max_new_tokens=80)
    json_responses.extend(resps)

import json

valid = 0
for i, resp in enumerate(json_responses):
    # Sometimes the model wraps JSON in markdown code blocks: strip those
    cleaned = resp.strip().strip('`').strip()
    if cleaned.startswith('json'):
        cleaned = cleaned[4:].strip()
    try:
        parsed = json.loads(cleaned)
        if 'label' in parsed and 'confidence' in parsed:
            valid += 1
            if i < 5:
                print(f"  [OK] {parsed}")
        else:
            if i < 5:
                print(f"  [PARTIAL] Parsed but missing fields: {parsed}")
    except json.JSONDecodeError:
        if i < 5:
            print(f"  [FAIL] Not valid JSON: {resp[:120]}")

print(f"\nValid JSON with correct fields: {valid} / {len(json_responses)} ({100*valid/len(json_responses):.0f}%)")
print("\n→ A 3B model will partially comply with JSON formatting. Larger models")
print("  (GPT-4o, Claude, Llama-70B) achieve near-perfect JSON compliance.")
print("  For production pipelines (Day 3), you need either a capable model or a")
print("  post-processing step that handles malformed outputs.")


### Exercise 2e: Chain-of-thought for hard cases

Some tweets are easy to classify. Others are ambiguous, sarcastic, or require reasoning about the author's intent. For these hard cases, asking the model to *explain its reasoning* before giving a label can improve accuracy.

This is **chain-of-thought (CoT) prompting**: instead of asking for a direct answer, you ask the model to think through the problem step by step.

Let's test this on the tweets that the model got wrong in Exercise 2a.


In [ ]:
#@title Identify misclassified tweets from zero-shot

# Find tweets where zero-shot prediction was wrong
wrong = val_data[val_data['pred_zero'] != val_data['stance_cat']].copy()
print(f"Zero-shot got {len(wrong)} tweets wrong out of {len(val_data)}.")
print(f"\nSample of misclassified tweets:")
for _, row in wrong.head(5).iterrows():
    print(f"  True: {row['stance_cat']}, Predicted: {row['pred_zero']}")
    print(f"  Text: {row['text_cleaned'][:120]}")
    print()


In [ ]:
### Exercise 2e: Chain-of-thought

# Write a CoT prompt. Ask the model to first explain what the author's position
# seems to be, THEN classify as support or oppose.
# Use {text} as the placeholder.

prompt_cot = ""  # YOUR CODE HERE

# Run on the misclassified tweets only
if prompt_cot and len(wrong) > 0:
    hard_subset = wrong.head(min(20, len(wrong)))  # cap at 20 to save time
    cot_responses = []
    messages = [prompt_cot.format(text=row['text_cleaned']) for _, row in hard_subset.iterrows()]
    for i in range(0, len(messages), 4):
        batch = messages[i:i+4]
        resps = generate_batch_instruct(
            instruct_model, instruct_tokenizer, batch, max_new_tokens=120
        )
        cot_responses.extend(resps)

    # Extract final labels
    cot_labels = []
    for resp in cot_responses:
        resp_lower = resp.lower()
        if "oppose" in resp_lower.split("\n")[-1] or resp_lower.strip().endswith("oppose"):
            cot_labels.append("oppose")
        elif "support" in resp_lower.split("\n")[-1] or resp_lower.strip().endswith("support"):
            cot_labels.append("support")
        elif "oppose" in resp_lower:
            cot_labels.append("oppose")
        elif "support" in resp_lower:
            cot_labels.append("support")
        else:
            cot_labels.append("unknown")

    # Compare
    true_labels = hard_subset['stance_cat'].tolist()
    direct_labels = hard_subset['pred_zero'].tolist()

    correct_direct = sum(1 for t, p in zip(true_labels, direct_labels) if t == p)
    correct_cot = sum(1 for t, p in zip(true_labels, cot_labels) if t == p)
    print(f"\nOn {len(hard_subset)} previously-misclassified tweets:")
    print(f"  Direct prompting: {correct_direct}/{len(hard_subset)} correct (these were all wrong by definition)")
    print(f"  Chain-of-thought:  {correct_cot}/{len(hard_subset)} correct")

    # Show a few examples with the model's reasoning
    print(f"\nSample CoT reasoning:")
    for i in range(min(3, len(cot_responses))):
        print(f"\n  True label: {true_labels[i]}")
        print(f"  Model reasoning: {cot_responses[i][:200]}")


In [ ]:
#@title Solution 2e

prompt_cot = (
    "Read the following tweet about the Women's March.\n\n"
    "Tweet: {text}\n\n"
    "First, explain in one sentence what the author's position on the Women's March seems to be. "
    "Then, on a new line, classify the tweet as: support or oppose.\n\n"
    "Reasoning:"
)

hard_subset = wrong.head(min(20, len(wrong)))
cot_responses = []
messages = [prompt_cot.format(text=row['text_cleaned']) for _, row in hard_subset.iterrows()]

for i in range(0, len(messages), 4):
    batch = messages[i:i+4]
    resps = generate_batch_instruct(
        instruct_model, instruct_tokenizer, batch, max_new_tokens=120
    )
    cot_responses.extend(resps)

cot_labels = []
for resp in cot_responses:
    resp_lower = resp.lower()
    # Try to get the label from the last line first
    last_line = resp_lower.strip().split("\n")[-1]
    if "oppose" in last_line:
        cot_labels.append("oppose")
    elif "support" in last_line:
        cot_labels.append("support")
    elif "oppose" in resp_lower:
        cot_labels.append("oppose")
    elif "support" in resp_lower:
        cot_labels.append("support")
    else:
        cot_labels.append("unknown")

true_labels = hard_subset['stance_cat'].tolist()
correct_cot = sum(1 for t, p in zip(true_labels, cot_labels) if t == p)
total = len(hard_subset)

print(f"On {total} previously-misclassified tweets:")
print(f"  Direct prompting: 0/{total} correct (all were wrong)")
print(f"  Chain-of-thought:  {correct_cot}/{total} correct")
print()

for i in range(min(3, len(cot_responses))):
    print(f"True: {true_labels[i]} | CoT predicted: {cot_labels[i]}")
    print(f"Reasoning: {cot_responses[i][:250]}")
    print()

print("→ CoT helps the model reason about ambiguous cases. When the tweet is sarcastic,")
print("  or when the author's position requires inference, step-by-step reasoning gives")
print("  the model a chance to work through the logic before committing to a label.")
print()
print("→ Frontier reasoning models (o1, DeepSeek-R1, Claude with extended thinking)")
print("  bake this process into the model itself: they 'think' before answering,")
print("  without needing explicit CoT prompting. The tradeoff is higher latency and cost.")


### Stretch exercise: Stance vs. sentiment (if time permits)

The Bestvater and Monroe paper makes a point that matters for every social scientist using LLMs: **what you ask for is what you get**, and if you ask for the wrong construct, you will measure the wrong thing.

Change your prompt from classifying **stance** (support/oppose) to classifying **sentiment** (positive/negative). Run it on the same tweets. Compare the sentiment predictions to the true *stance* labels.


In [ ]:
### Stretch: Stance vs. Sentiment

prompt_sentiment = (
    "What is the sentiment of the following tweet? "
    "Answer with one word: positive or negative.\n\n"
    "Tweet: {text}\n"
    "Answer:"
)

preds_sentiment = run_classification(
    instruct_model, instruct_tokenizer, val_data,
    prompt_template=prompt_sentiment,
    batch_size=8,
    labels=("positive", "negative")
)

# Compare sentiment predictions to TRUE STANCE labels
# (not sentiment labels: that's the point)
preds_mapped = ["support" if p == "positive" else "oppose" if p == "negative" else "unknown"
                for p in preds_sentiment]

print("If sentiment = stance, this should match our best classification results.")
print("It probably won't.\n")
print("Sentiment predictions vs. true STANCE labels:")
print(classification_report(val_data['stance_cat'], preds_mapped, digits=3))

print("\nCompare to our zero-shot STANCE classification:")
print(classification_report(val_data['stance_cat'], val_data['pred_zero'], digits=3))

print("→ Sentiment and stance are different constructs. A tweet can support the")
print("  Women's March in an angry tone (positive stance, negative sentiment), or")
print("  oppose it politely (negative stance, positive sentiment).")
print("  Your prompt defines your construct. Get it wrong, and you measure the wrong thing.")


**Section takeaway: Prompting is experimental design.** Your prompt determines what you measure (construct validity), how reliably you measure it (reproducibility), and whether additional complexity (few-shot, CoT) is worth the cost. Every choice, from wording to label names to prompt structure, is a methodological decision. Treat prompt design with the same rigor you would apply to survey question design: justify your choices, test for sensitivity, and report what you tried.


---

# Section 3: The Model Landscape and Bridge to Day 3

You have been working with a single 3-billion-parameter model. There are hundreds of models available, ranging from 1B to over 400B parameters, open and closed. How do you choose?


## Benchmarks: what they measure and what they miss

The most common model benchmarks:

| Benchmark | What it measures | Limitation |
|-----------|-----------------|------------|
| **MMLU** | Broad knowledge across 57 subjects | Models may have memorized test questions |
| **HumanEval** | Code generation ability | Only measures Python, narrow task |
| **GPQA** | Expert-level reasoning (graduate-level science) | Very small test set |
| **Chatbot Arena** | Human preference in open-ended conversation | Reflects general users, not researchers |

**Key limitations of all benchmarks:**

- **Contamination:** models may have seen benchmark questions during training, inflating scores.
- **Gaming:** providers can optimize specifically for benchmarks without improving general ability.
- **Construct validity:** a high MMLU score does not mean the model will classify your political texts well. No benchmark measures *your* task. That is why building task-specific evaluations (which you will do in Day 3) matters more than any leaderboard.

The most ecologically valid benchmark is arguably [Chatbot Arena](https://huggingface.co/spaces/lmarena-ai/chatbot-arena), which ranks models based on blind human preference comparisons on real conversations.


## Open vs. closed models

| | Open models | Closed models |
|---|---|---|
| **Examples** | Llama, Qwen, Mistral, Gemma | GPT-4o, Claude, Gemini |
| **Capability** | Catching up; frontier open models now rival closed | Still ahead on hardest tasks |
| **Cost** | Free to run locally; pay for compute | Pay per token |
| **Transparency** | Full access to weights, can inspect and modify | Black box |
| **Data privacy** | Your data never leaves your machine | Data sent to provider's API |
| **Reproducibility** | Weights are fixed; same model forever | Provider can update silently |

**For social science research:**

- If your data is sensitive (survey responses, medical records, classified documents), open models let you process everything locally.
- If you need maximum capability and your data is not sensitive, closed models are currently ahead.
- If reproducibility matters (it should), open models are safer: a closed model can change between when you run your study and when reviewers try to replicate it.
- The gap between open and closed models is narrowing rapidly.

## What you just experienced

You worked with a 3B-parameter open model. It classified political tweets reasonably well but struggled with JSON formatting and some ambiguous cases. A frontier model (50--100x larger, with extensive post-training) would likely perform better on all of these tasks. But it costs money per token, you cannot run it locally, and you cannot inspect its internals. That tradeoff is real, and there is no universal right answer.


### Exercise 3a: Save your work (Bridge to Day 3)

Pick your best prompt from Section 2 (whichever gave the best results). Save three things:

1. The prompt template you used
2. The model's predictions on all 250 tweets
3. Your own manual labels for 10 tweets (you will label them yourself right now)

Day 3 opens by loading this file and computing inter-annotator agreement between you and the model.


In [ ]:
### Exercise 3a: Save your work

# 1. Pick your best prompt (copy it from the exercise that worked best)
best_prompt = ""  # YOUR CODE HERE

# 2. The predictions are already saved in val_data. Pick the best column:
best_pred_column = "pred_zero"  # change to "pred_few", "pred_v1", etc.

# 3. Manually label 10 tweets. Read each one and decide: support or oppose.
#    (This takes about 5 minutes. Be honest: some tweets are ambiguous.)

sample_for_labeling = val_data.head(10)
print("Label each tweet as 'support' or 'oppose':\n")
for i, (_, row) in enumerate(sample_for_labeling.iterrows()):
    print(f"[{i}] {row['text_cleaned'][:150]}")
    print(f"    Model said: {row[best_pred_column]}")
    print()

# Fill in your labels below (list of 10 strings: "support" or "oppose")
my_labels = [
    "",  # tweet 0
    "",  # tweet 1
    "",  # tweet 2
    "",  # tweet 3
    "",  # tweet 4
    "",  # tweet 5
    "",  # tweet 6
    "",  # tweet 7
    "",  # tweet 8
    "",  # tweet 9
]


In [ ]:
#@title Save results to CSV

# Save the validation data with all predictions and your manual labels
output = val_data[['text_cleaned', 'stance_cat', 'sentiment_cat']].copy()
output['model_prediction'] = val_data[best_pred_column]
output['prompt_used'] = best_prompt

# Add manual labels for the first 10
output['human_label'] = None
for i, label in enumerate(my_labels):
    if label:
        output.loc[output.index[i], 'human_label'] = label

output.to_csv('day2_classification_results.csv', index=False)
print("Saved to day2_classification_results.csv")
print(f"Rows: {len(output)}")
print(f"Rows with human labels: {output['human_label'].notna().sum()}")
print("\n→ You will load this file at the start of Day 3 to compute")
print("  inter-annotator agreement (Cohen's kappa) and build a validation pipeline.")


---

# Closing

## What you learned today

1. **Post-training** turns base models into useful tools. SFT teaches format, preference optimization teaches quality, safety training teaches boundaries. These are design choices that affect your research.

2. **Prompting is experimental design.** Your prompt wording determines what you measure and how reliably. You saw 5--15% swings in classification accuracy from minor rephrasing.

3. **Progressive techniques** (few-shot, structured output, chain-of-thought) each have their place. Few-shot helps when the model needs calibration. Structured output enables pipelines. CoT helps with ambiguous cases. None is universally best.

4. **Model choice** involves real tradeoffs between capability, cost, transparency, and reproducibility. There is no single "best model."

| Day | Topic | Status |
|-----|-------|--------|
| 1 | From Embeddings to Transformers | ✓ |
| 2 | From Models to Tools | ✓ |
| **→ 3** | **Deploying for Research** | **Next** |
| 4 | Social Science Applications | |
| 5 | Agentic Workflows | |

## Bridge to Day 3

You now know how to prompt models and you have seen how sensitive the results are to design choices. Day 3 takes the next step: how to build rigorous classification pipelines at scale (APIs, batching, cost management), how to validate your outputs (inter-annotator agreement, gold-standard sets), and when prompting is not enough and you need to fine-tune.

You will start Day 3 by loading the CSV you just saved and computing agreement metrics on it.

---

Course website: [llmsforsocialscience.net](https://llmsforsocialscience.net)
